# Project Frontier IA — Pipeline Runner

Este notebook baixa o projeto, configura um **distrobox** com todas as dependências (Node.js, Python, TensorFlow, PyTorch, ONNX), e executa o `run_pipeline` end-to-end. Todos os logs são capturados e exibidos nas células.

**Pré-requisitos na máquina host:**
- `distrobox` instalado
- `podman` ou `docker` instalado
- Jupyter/IPython para rodar este notebook

## 1. Configuração do Distrobox

Cria um container distrobox com Ubuntu 24.04, instala Node.js 20.x, Python 3.12, e todas as dependências do projeto.

In [ ]:
import subprocess
import sys
import os

# Nome do container distrobox
BOX_NAME = "frontier-ia"
# Diretório onde o projeto será clonado DENTRO do container
PROJECT_DIR = "/home/frontier/Project-Frontier-IA"
# Repo URL
REPO_URL = "https://github.com/NotAdson/Project-Frontier-IA.git"
# Branch
BRANCH = "dev"

def run(cmd, shell=False, check=True, capture=False, timeout=None):
    """Executa um comando e retorna o resultado. Se capture=True, captura stdout/stderr."""
    if shell:
        cmd_str = cmd
    else:
        cmd_str = " ".join(cmd) if isinstance(cmd, list) else cmd
    print(f"$ {cmd_str}")
    if capture:
        result = subprocess.run(cmd, shell=shell, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0 and check:
            print(f"STDERR: {result.stderr}")
            raise RuntimeError(f"Command failed: {cmd_str}")
        return result
    else:
        result = subprocess.run(cmd, shell=shell, timeout=timeout)
        if result.returncode != 0 and check:
            raise RuntimeError(f"Command failed: {cmd_str}")
        return result

def box_exec(cmd, capture=False, timeout=None, check=True):
    """Executa um comando dentro do distrobox."""
    full_cmd = ["distrobox", "exec", "--name", BOX_NAME, "--", "bash", "-lc", cmd]
    return run(full_cmd, capture=capture, timeout=timeout, check=check)

print("Funções auxiliares definidas.")

## 2. Criar o container Distrobox

Se o container já existir, pula a criação.

In [ ]:
# Verifica se o container já existe
result = run(["distrobox", "list"], capture=True, check=False)
existing_boxes = [l.split()[0] for l in result.stdout.strip().split("\n")[1:] if l.strip()]

if BOX_NAME in existing_boxes:
    print(f"Container '{BOX_NAME}' já existe. Pulando criação.")
else:
    print(f"Criando container '{BOX_NAME}'...")
    run([
        "distrobox", "create",
        "--name", BOX_NAME,
        "--image", "ubuntu:24.04",
        "--additional-flags", "-v /tmp/frontier-logs:/tmp/frontier-logs",
        "--yes",
    ])
    print("Container criado.")

## 3. Instalar dependências do sistema

Node.js 20.x, Python 3.12, git, build-essential.

In [ ]:
# Instala Node.js 20.x via NodeSource
box_exec("""
apt-get update && apt-get install -y curl git build-essential python3 python3-pip python3-venv
curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get install -y nodejs
node --version
npm --version
python3 --version
""")

## 4. Clonar o repositório

In [ ]:
# Clona o repo se não existir ainda
result = box_exec(f"test -d {PROJECT_DIR} && echo EXISTS || echo MISSING", capture=True, check=False)
if "EXISTS" in result.stdout:
    print(f"Projeto já clonado em {PROJECT_DIR}. Pulando clone.")
    # Faz pull para atualizar
    box_exec(f"cd {PROJECT_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
else:
    print(f"Clonando projeto em {PROJECT_DIR}...")
    box_exec(f"mkdir -p /home/frontier && cd /home/frontier && git clone --branch {BRANCH} {REPO_URL} Project-Frontier-IA")
    print("Clone concluído.")

## 5. Buildar o engine (Node.js)

O engine Pokémon Showdown precisa ser compilado com `npm install` e `npm run build`.

In [ ]:
box_exec(f"""
cd {PROJECT_DIR}/engine
npm install
npm run build
ls dist/sim/battle.js dist/sim/teams.js
""")

## 6. Instalar dependências Python

Cria um venv e instala todas as dependências do `requirements.txt` + TensorFlow + PyTorch.

In [ ]:
box_exec(f"""
cd {PROJECT_DIR}
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
pip install tensorflow
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install jupyter ipykernel
python -c "import keras; import tensorflow as tf; import torch; import onnx; print('All imports OK')"
""")

## 7. Baixar os times do Metamon

Os times competitivos Gen 3 OU precisam ser baixados do HuggingFace.

In [ ]:
box_exec(f"""
cd {PROJECT_DIR}
source .venv/bin/activate
python scripts/download_metamon_teams.py
wc -l data/teams/gen3ou.txt
head -1 data/teams/gen3ou.txt | cut -c1-100
""")

## 8. Executar o Pipeline

Executa o `run_pipeline.py` end-to-end. Todos os logs são salvos em `/tmp/frontier-logs/pipeline.log` e exibidos na célula.

**Parâmetros atuais:**
- `num_games=500` — jogos de self-play por geração
- `num_generations=1000` — número de gerações
- `mcts_iterations=300` — iterações MCTS por move
- `epochs=100` — épocas de treino por geração
- `max_rollout_depth=10` — profundidade máxima do rollout MCTS

Ajuste conforme necessário.

In [ ]:
# Parâmetros do pipeline
NUM_GAMES = 500
NUM_GENERATIONS = 1000
MCTS_ITERATIONS = 300
EPOCHS = 100
ROLLOUT_DEPTH = 10

# Comando do pipeline
pipeline_cmd = f"""
cd {PROJECT_DIR}
source .venv/bin/activate
mkdir -p /tmp/frontier-logs
python src/battle_agents/mcts_approximation/pipeline/run_pipeline.py \
  --num-games {NUM_GAMES} \
  --num-generations {NUM_GENERATIONS} \
  --mcts-iterations {MCTS_ITERATIONS} \
  --epochs {EPOCHS} \
  --max-rollout-depth {ROLLOUT_DEPTH} \
  2>&1 | tee /tmp/frontier-logs/pipeline.log
"""

print("=== Iniciando pipeline ===")
print(f"Parâmetros: games={NUM_GAMES}, gens={NUM_GENERATIONS}, mcts={MCTS_ITERATIONS}, epochs={EPOCHS}, rollout_depth={ROLLOUT_DEPTH}")
print(f"Log salvo em: /tmp/frontier-logs/pipeline.log")
print()

# Executa o pipeline (sem timeout — pode demorar horas)
box_exec(pipeline_cmd, check=False)

## 9. Verificar resultados

Inspeciona os modelos gerados, logs de treino, e relatórios de benchmark.

In [ ]:
# Lista os modelos e dados gerados
box_exec(f"""
cd {PROJECT_DIR}
echo "=== Modelos gerados ==="
ls -la data/mcts_model.* 2>/dev/null || echo "Nenhum modelo na raiz"
echo
echo "=== Gerações ==="
ls -d data/gen* 2>/dev/null || echo "Nenhuma geração"
echo
echo "=== Champion ==="
cat data/champion.json 2>/dev/null || echo "Nenhum champion"
echo
echo "=== Autoencoder checkpoint ==="
ls -la data/autoencoder_bootstrap/checkpoints_v5_fixed256/fused_autoencoder_best.pt 2>/dev/null || echo "Autoencoder não treinado"
echo
echo "=== Últimas 50 linhas do log ==="
tail -50 /tmp/frontier-logs/pipeline.log 2>/dev/null || echo "Log não encontrado"
""", check=False)

## 10. (Opcional) Executar o app web

Inicia o servidor Flask para jogar contra a IA pelo navegador.

In [ ]:
# Descomente para rodar o app web
# box_exec(f"""
# cd {PROJECT_DIR}
# source .venv/bin/activate
# python src/web/app.py 2>&1 | tee /tmp/frontier-logs/webapp.log
# """, check=False)

## 11. Limpar o container (opcional)

Remove o container distrobox e todos os dados.

In [ ]:
# Descomente para remover o container
# run(["distrobox", "rm", "--name", BOX_NAME, "--force"], check=False)
# print(f"Container '{BOX_NAME}' removido.")